# IsoTracks quickstart

One gene from an IsoForge run and its BAMs. Edit the paths in the first two code
cells, then run the notebook top to bottom.

In [ ]:
import os

import matplotlib.pyplot as plt

import isotracks
from isotracks import FusionView, IsoTracks, Layout, theme

print("IsoTracks", isotracks.__version__)

## 1. Inputs

`ISOFORGE` is the prefix IsoForge was run with (`isoforge annotate --o results/project`).
IsoTracks finds the transcript models, the read annotation and the TSS and
poly(A) clusters from it. BAMs are grouped by condition; replicates are pooled.

In [ ]:
ISOFORGE = "results/project"
BAM_DIR = "alignments"
OUTPUT_DIR = "figures"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BAM_FILES = {
    "Control": [f"{BAM_DIR}/control_rep{i}.bam" for i in (1, 2, 3)],
    "Treated": [f"{BAM_DIR}/treated_rep{i}.bam" for i in (1, 2, 3)],
}

## 2. Width and type size

Set once. `width` is the printed width (`single`, `onehalf`, `double`, `slide`
or millimetres).

In [ ]:
theme.use(width="single", readability="comfortable")

## 3. A gene

`min_reads_per_condition` keeps isoforms with at least that many reads in every
condition. An output name with no extension writes a PDF and a PNG.

In [ ]:
gene = "SAT1"

view = IsoTracks(
    gene_name=gene,
    isoforge=ISOFORGE,
    bam_files=BAM_FILES,
    min_reads_per_condition=5,
)
view.plot(output=f"{OUTPUT_DIR}/{gene}")
plt.show()

## 4. The 3' end

`zoom` returns a window: `first_exon`, `last_exon`, `exon_<N>`, `5_prime`,
`3_prime` or `whole_gene`. `sort_style="diamond"` centres the longest tails.

In [ ]:
window = view.zoom("last_exon", padding=400)

view.plot(region=window, output=f"{OUTPUT_DIR}/{gene}_3prime")
view.plot(region=window, sort_style="diamond", output=f"{OUTPUT_DIR}/{gene}_3prime_diamond")
plt.show()

## 5. A close-up beside the gene, and shortened introns

In [ ]:
view.plot(detail="last_exon", output=f"{OUTPUT_DIR}/{gene}_detail")
view.plot(introns="scaled", output=f"{OUTPUT_DIR}/{gene}_scaled")
plt.show()

## 6. Coordinates and tail lengths as tables

In [ ]:
view.print_info()

table = view.polya_table()
table.groupby(["condition", "transcript_id"])["polya_length"].agg(["count", "median"])

## 7. A gene list

In [ ]:
for name in ["SAT1", "ACTB", "GAPDH"]:
    try:
        IsoTracks(gene_name=name, isoforge=ISOFORGE, bam_files=BAM_FILES,
                  min_reads_per_condition=5, verbose=False).plot(output=f"{OUTPUT_DIR}/{name}")
    except Exception as error:
        print(f"{name}: skipped ({error})")
    finally:
        plt.close("all")

## 8. A fusion

By its genes or its call ID in the run's fusion tables. A reference GTF adds each
partner's whole gene to the schematic.

In [ ]:
fusion = FusionView("RIMS2::ATP6V1C1", isoforge=ISOFORGE, bam_files=BAM_FILES,
                    reference_gtf="gencode.gtf")
fusion.plot(output=f"{OUTPUT_DIR}/RIMS2_ATP6V1C1")
plt.show()

## 9. Colours and track heights

In [ ]:
custom = IsoTracks(
    gene_name=gene,
    isoforge=ISOFORGE,
    bam_files=BAM_FILES,
    condition_colors={"Control": "#9AA5B1", "Treated": "#1B2A49"},
    isoform_colors=["#0072B2", "#D55E00", "#009E73", "#CC79A7"],
    layout=Layout(read_stack=4.0),
)
custom.plot(width="double", output=f"{OUTPUT_DIR}/{gene}_wide")
plt.show()